# Step 1: Extract MARCXML to Relational CSVs

## Overview
This notebook implements the data extraction phase of the KBR Google Books digitization project analysis pipeline. 
Input: 1,000 historical bibliographic records in MARCXML format from KBR.
Output: 7 normalized relational CSV files respecting 3NF principles.

**Key Design Decisions:**
- Year: Extracted strictly from MARC field `008` positions 07–10 (primary publication year only)
- Language: First occurrence of `041 $a`; multiple languages grouped as "multiple languages"
- Cities, Countries, Publishers: Multi-valued extraction using bridge tables
- Publishers: Authority-controlled via fields 100/110/700/710/720 where $4="pbl", bypassing raw `264 $b`

## Setup: Imports and Configuration

In [1]:
import xml.etree.ElementTree as ET
import pandas as pd
import os
from collections import defaultdict

# Configure display options
pd.set_option('display.max_rows', 10)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

print("Libraries loaded successfully.")

Libraries loaded successfully.


## Load MARCXML Input Data

In [18]:
# Define input and output paths
xml_file = '../data/raw/0506_Q1_metadata.xml'  # Adjust path as needed
output_dir = '../data/raw/'

# Create output directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

# Parse MARCXML
try:
    tree = ET.parse(xml_file)
    root = tree.getroot()
    records = root.findall('record')
    print(f"✓ MARCXML loaded successfully.")
    print(f"✓ Total bibliographic records: {len(records)}")
except FileNotFoundError:
    print(f"ERROR: File not found at {xml_file}")
except Exception as e:
    print(f"ERROR: Failed to parse XML - {e}")

✓ MARCXML loaded successfully.
✓ Total bibliographic records: 1000


## Initialize Data Structures
Prepare lists to collect extracted data for each entity type.

In [37]:
# Initialize lists for storing extracted data
books_data = []
book_cities_data = []
cities_set = set()

book_countries_data = []
countries_set = set()

book_publishers_data = []
publishers_set = set()

# Tracking
extraction_stats = {
    'total_records': len(records),
    'records_with_year': 0,
    'records_with_language': 0,
    'records_with_cities': 0,
    'records_with_countries': 0,
    'records_with_publishers': 0,
    'multi_language_records': 0
}

print("Data structures initialized.")

Data structures initialized.


## Extract Data from MARCXML Records
Iterate through all records and extract required fields according to the MARC 21 standard.

In [38]:
for record_idx, record in enumerate(records, start=1):
    # Extract Book ID from MARC field 001 (Control Number / IDN)
    control_number = record.find(".//controlfield[@tag='001']")
    if control_number is None:
        print(f"Warning: Record {record_idx} has no control number. Skipping.")
        continue
    
    book_id = control_number.text.strip()
    
    # ===== EXTRACT TITLE (MARC 245) =====
    title_field = record.find(".//datafield[@tag='245']")
    title = None
    if title_field is not None:
        title_subfield = title_field.find("subfield[@code='a']")
        if title_subfield is not None:
            title = title_subfield.text.strip() if title_subfield.text else None
    
    # # ===== EXTRACT YEAR (MARC 008, positions 07–10) =====
    # year = None
    # control_field_008 = record.find(".//controlfield[@tag='008']")
    # if control_field_008 is not None and control_field_008.text:
    #     # MARC 008 is a fixed field; positions 07–10 contain Date 1
    #     try:
    #         year_str = control_field_008.text[7:11].strip()
    #         if year_str and year_str != '    ':  # Check for non-empty and non-space
    #             year = int(year_str)
    #             extraction_stats['records_with_year'] += 1
    #     except (ValueError, IndexError):
    #         pass  # Invalid year format; leave as None
    
    # ===== EXTRACT YEAR (MARC 008 positions 07–10, supplemented by 264 $c) =====
    year = None
    control_field_008 = record.find(".//controlfield[@tag='008']")

    # Priority 1: Extract from MARC 008 positions 07–10
    if control_field_008 is not None and control_field_008.text:
        try:
            year_str = control_field_008.text[7:11].strip()
            if year_str and year_str != '    ':
                year = int(year_str)
                extraction_stats['records_with_year'] += 1
        except (ValueError, IndexError):
            pass

    # Priority 2: If 008 fails, try MARC 264 $c (Publication date)
    if year is None:
        for field_264 in record.findall(".//datafield[@tag='264']"):
            date_sf = field_264.find("subfield[@code='c']")
            if date_sf is not None and date_sf.text:
                # Extract first 4 digits from date string (e.g., "1855", "1860-1862" → 1860)
                import re
                date_match = re.search(r'(\d{4})', date_sf.text)
                if date_match:
                    try:
                        year = int(date_match.group(1))
                        extraction_stats['records_with_year'] += 1
                        break  # Use first match only
                    except ValueError:
                        pass
    
    # ===== EXTRACT LANGUAGE (MARC 041 $a) =====
    language = None
    lang_field = record.find(".//datafield[@tag='041']")
    if lang_field is not None:
        lang_subfields = lang_field.findall("subfield[@code='a']")
        if lang_subfields:
            if len(lang_subfields) > 1:
                language = "multiple languages"
                extraction_stats['multi_language_records'] += 1
            else:
                language = lang_subfields[0].text.strip() if lang_subfields[0].text else None
            if language:
                extraction_stats['records_with_language'] += 1
    
    # ===== EXTRACT CITIES (MARC 264 $a, multi-valued) =====
    cities = []
    for field_264 in record.findall(".//datafield[@tag='264']"):
        city_subfields = field_264.findall("subfield[@code='a']")
        for city_sf in city_subfields:
            if city_sf.text:
                city = city_sf.text.strip()
                if city:
                    cities.append(city)
                    cities_set.add(city)
    
    if cities:
        extraction_stats['records_with_cities'] += 1
    
    # ===== EXTRACT COUNTRIES (MARC 044 $a, multi-valued) =====
    countries = []
    for field_044 in record.findall(".//datafield[@tag='044']"):
        country_subfields = field_044.findall("subfield[@code='a']")
        for country_sf in country_subfields:
            if country_sf.text:
                country = country_sf.text.strip()
                if country:
                    countries.append(country)
                    countries_set.add(country)
    
    if countries:
        extraction_stats['records_with_countries'] += 1
    
    # ===== EXTRACT PUBLISHERS (MARC 100, 110, 700, 710, 720 where $4="pbl") =====
    publishers = []
    for tag in ['100', '110', '700', '710', '720']:
        for field in record.findall(f".//datafield[@tag='{tag}']"):
            # Check if $4 (Relator Code) contains 'pbl' (Publisher)
            role_subfields = field.findall("subfield[@code='4']")
            roles = [sf.text for sf in role_subfields if sf.text]
            
            if 'pbl' in roles:
                # Extract Name ($a) when role is publisher
                name_sf = field.find("subfield[@code='a']")
                if name_sf is not None and name_sf.text:
                    publisher = name_sf.text.strip()
                    publishers.append({
                        'publisher_name': publisher,
                        'source_field': tag
                    })
                    publishers_set.add(publisher)
    
    if publishers:
        extraction_stats['records_with_publishers'] += 1
    
    # ===== AGGREGATE EXTRACTED DATA =====
    # Add to books table
    books_data.append({
        'book_id': book_id,
        'title': title,
        'year': year,
        'language': language
    })
    
    # Add book-city relationships
    for city in cities:
        book_cities_data.append({
            'book_id': book_id,
            'city_name': city
        })
    
    # Add book-country relationships
    for country in countries:
        book_countries_data.append({
            'book_id': book_id,
            'country_code': country
        })
    
    # Add book-publisher relationships
    for pub_info in publishers:
        book_publishers_data.append({
            'book_id': book_id,
            'publisher_name': pub_info['publisher_name'],
            'source_field': pub_info['source_field']
        })

print("✓ Extraction complete. Records processed: {}/{}".format(len(books_data), extraction_stats['total_records']))

✓ Extraction complete. Records processed: 1000/1000


## Extraction Statistics

In [39]:
print("\n=== EXTRACTION STATISTICS ===")
print(f"Total records processed: {extraction_stats['total_records']}")
print(f"Records with year: {extraction_stats['records_with_year']}")
print(f"Records with language: {extraction_stats['records_with_language']}")
print(f"  - Multi-language records: {extraction_stats['multi_language_records']}")
print(f"Records with cities: {extraction_stats['records_with_cities']}")
print(f"  - Total city relationships: {len(book_cities_data)}")
print(f"  - Unique cities: {len(cities_set)}")
print(f"Records with countries: {extraction_stats['records_with_countries']}")
print(f"  - Total country relationships: {len(book_countries_data)}")
print(f"  - Unique countries: {len(countries_set)}")
print(f"Records with publishers: {extraction_stats['records_with_publishers']}")
print(f"  - Total publisher relationships: {len(book_publishers_data)}")
print(f"  - Unique publishers: {len(publishers_set)}")


=== EXTRACTION STATISTICS ===
Total records processed: 1000
Records with year: 988
Records with language: 1000
  - Multi-language records: 13
Records with cities: 1000
  - Total city relationships: 1073
  - Unique cities: 270
Records with countries: 1000
  - Total country relationships: 1014
  - Unique countries: 30
Records with publishers: 452
  - Total publisher relationships: 476
  - Unique publishers: 376


## Create DataFrames and Inspect

In [41]:
# Create DataFrames from collected data
df_books = pd.DataFrame(books_data)
df_book_cities = pd.DataFrame(book_cities_data)
df_cities = pd.DataFrame({'city_name': list(cities_set)})
df_book_countries = pd.DataFrame(book_countries_data)
df_countries = pd.DataFrame({'country_code': list(countries_set)})
df_book_publishers = pd.DataFrame(book_publishers_data)
df_publishers = pd.DataFrame({'publisher_name': list(publishers_set)})
df_books['year'] = df_books['year'].astype('Int64')  

print(df_books['year'].dtype) 
print(df_books['year'].isnull().sum())

print("✓ DataFrames created.")
print(f"\ndf_books shape: {df_books.shape}")
print(df_books.head())

Int64
12
✓ DataFrames created.

df_books shape: (1000, 4)
    book_id                                              title  year language
0  12163581  Obst-und Trauben-Ausstellung Die Württembergis...  1852      ger
1  11528967                                         Anneessens  1870      fre
2  12021785                             The Austin-Topolovampo  1875      eng
3  22169951                                          La Logica  1880      ita
4  11677526                      Économie politique et sociale  1865      fre


In [42]:
print(f"\ndf_book_cities shape: {df_book_cities.shape}")
print(df_book_cities.head())
print(f"\ndf_cities shape: {df_cities.shape}")
print(df_cities.head(10))


df_book_cities shape: (1073, 2)
    book_id                              city_name
0  12163581                              Stuttgart
1  11528967                              Bruxelles
2  12021785  [Place of publication not identified]
3  22169951                                 Torino
4  11677526                              Bruxelles

df_cities shape: (270, 1)
           city_name
0  Santiago de Chilé
1              Halle
2             Brugge
3          Amsterdam
4  Chalons-sur-Marne
5         Purmerende
6         Strasbourg
7           Courtrai
8            Avignon
9            Valence


In [43]:
print(f"\ndf_book_countries shape: {df_book_countries.shape}")
print(df_book_countries.head())
print(f"\ndf_countries shape: {df_countries.shape}")
print(df_countries.sort_values('country_code'))


df_book_countries shape: (1014, 2)
    book_id country_code
0  12163581           gw
1  11528967           be
2  12021785           xx
3  22169951           it
4  11677526           be

df_countries shape: (30, 1)
   country_code
4            ag
24           au
11           be
10           cl
2            dk
..          ...
7            uy
18           xr
25           xx
28          xxk
6           xxu

[30 rows x 1 columns]


In [44]:
print(f"\ndf_book_publishers shape: {df_book_publishers.shape}")
print(df_book_publishers.head())
print(f"\ndf_publishers shape: {df_publishers.shape}")
print(df_publishers.head(10))


df_book_publishers shape: (476, 3)
    book_id                publisher_name source_field
0  12163581                Blum und Bogel          720
1  13672019     Giard, V., & Emile Brière          710
2  11338323  Dela Montagne, Victor Alexis          100
3  11338323                  Mienikus, M.          700
4  11484566    Bruylant-Christophe et Cie          710

df_publishers shape: (376, 1)
                                      publisher_name
0                                   Vleminckx, Henri
1       E. Plon, Nourrit et Cie, imprimeurs-éditeurs
2                           Buschmann, Joseph-Ernest
3                            Marchands de nouveautés
4                                 Librairie Nouvelle
5                  El Mensajero del corazon de Jesus
6                               Jouret et Thémon, E.
7  Kommissionsverlag der Fr. Lintz'schen Buchhand...
8                         C. Marpon et E. Flammarion
9                                      Perrin et Cie


## Data Quality Checks

In [45]:
# Check for null values
print("=== NULL VALUE CHECK ===")
print("\ndf_books:")
print(df_books.isnull().sum())

print("\ndf_cities (all entries should be non-null):")
print(df_cities.isnull().sum())

print("\ndf_countries (all entries should be non-null):")
print(df_countries.isnull().sum())

print("\ndf_publishers (all entries should be non-null):")
print(df_publishers.isnull().sum())

=== NULL VALUE CHECK ===

df_books:
book_id      0
title        0
year        12
language     0
dtype: int64

df_cities (all entries should be non-null):
city_name    0
dtype: int64

df_countries (all entries should be non-null):
country_code    0
dtype: int64

df_publishers (all entries should be non-null):
publisher_name    0
dtype: int64


In [46]:
# Check for duplicates
print("\n=== DUPLICATE CHECK ===")
print(f"Duplicate cities: {df_cities['city_name'].duplicated().sum()}")
print(f"Duplicate countries: {df_countries['country_code'].duplicated().sum()}")
print(f"Duplicate publishers: {df_publishers['publisher_name'].duplicated().sum()}")


=== DUPLICATE CHECK ===
Duplicate cities: 0
Duplicate countries: 0
Duplicate publishers: 0


## Export to CSV Files
Write all 7 relational CSV files to the output directory.

In [47]:
# Define output file paths
csv_files = {
    'books.csv': df_books,
    'book_cities.csv': df_book_cities,
    'cities.csv': df_cities,
    'book_countries.csv': df_book_countries,
    'countries.csv': df_countries,
    'book_publishers.csv': df_book_publishers,
    'publishers.csv': df_publishers
}

# Write each CSV
for filename, df in csv_files.items():
    filepath = os.path.join(output_dir, filename)
    df.to_csv(filepath, index=False, encoding='utf-8')
    print(f"✓ {filename:30s} ({df.shape[0]} rows, {df.shape[1]} columns)")

print(f"\n✓ All CSV files exported to: {output_dir}")

✓ books.csv                      (1000 rows, 4 columns)
✓ book_cities.csv                (1073 rows, 2 columns)
✓ cities.csv                     (270 rows, 1 columns)
✓ book_countries.csv             (1014 rows, 2 columns)
✓ countries.csv                  (30 rows, 1 columns)
✓ book_publishers.csv            (476 rows, 3 columns)
✓ publishers.csv                 (376 rows, 1 columns)

✓ All CSV files exported to: ../data/raw/


## Summary
Step 1 extraction is complete. The pipeline has produced 7 normalized relational CSV files:

1. **books.csv** - Core bibliographic records (book_id, title, year, language)
2. **book_cities.csv** - Book-city relationships (bridge table)
3. **cities.csv** - Deduplicated city entities (for OpenRefine cleaning)
4. **book_countries.csv** - Book-country relationships (bridge table)
5. **countries.csv** - Deduplicated country codes (for Step 3 mapping)
6. **book_publishers.csv** - Book-publisher relationships (bridge table, includes source_field)
7. **publishers.csv** - Deduplicated publisher entities (for OpenRefine cleaning)

**Next Step:** Proceed to Step 2 (Clean) to apply syntactic standardization and semantic enrichment via OpenRefine and Wikidata reconciliation.

In [48]:
print("\n" + "="*60)
print("STEP 1 EXTRACTION COMPLETE")
print("="*60)
print(f"\nOutput directory: {output_dir}")
print(f"Ready for Step 2: Data Cleaning (OpenRefine + Python Preprocessing)")


STEP 1 EXTRACTION COMPLETE

Output directory: ../data/raw/
Ready for Step 2: Data Cleaning (OpenRefine + Python Preprocessing)


In [49]:
## Publisher Coverage Analysis
# This section identifies books without publisher information
# for manual verification against the KBR backend database

import os
print(f"Current working directory: {os.getcwd()}")

import pandas as pd

# Load the books and book_publishers data
df_books = pd.read_csv('../data/raw/books.csv')
df_book_publishers = pd.read_csv('../data/raw/book_publishers.csv')

# Identify books that have publisher records
books_with_publisher = set(df_book_publishers['book_id'].unique())

# Filter for books WITHOUT publisher information
books_without_publisher = df_books[~df_books['book_id'].isin(books_with_publisher)]

print(f"\n=== PUBLISHER COVERAGE VERIFICATION ===")
print(f"Total books in dataset: {len(df_books)}")
print(f"Books with authority-controlled publishers: {len(books_with_publisher)}")
print(f"Books WITHOUT publishers: {len(books_without_publisher)}")
print(f"Coverage rate: {(len(books_with_publisher) / len(df_books) * 100):.1f}%")

print(f"\n=== SAMPLE: Books without publisher (first 20) ===")
print(books_without_publisher[['book_id', 'title']].head(20))

# Export the complete list for manual verification at KBR backend
output_file = '../data/raw/books_without_publisher_verification.csv'
books_without_publisher[['book_id', 'title']].to_csv(output_file, index=False)
print(f"\n✓ Exported verification list to: {output_file}")
print(f"  Use this list to cross-check against KBR backend records")

Current working directory: /Users/sophiewong/Documents/2025_KUL/2026_Thesis/thesis-kbr-google-books/notebooks

=== PUBLISHER COVERAGE VERIFICATION ===
Total books in dataset: 1000
Books with authority-controlled publishers: 452
Books WITHOUT publishers: 548
Coverage rate: 45.2%

=== SAMPLE: Books without publisher (first 20) ===
     book_id                                 title
1   11528967                            Anneessens
2   12021785                The Austin-Topolovampo
3   22169951                             La Logica
4   11677526         Économie politique et sociale
8   13249372            Mes souvenirs sur Napoléon
..       ...                                   ...
23  22189337     Les puits artésiens de la Flandre
24  11488969      The adventures of my grandfather
25  13398939         Revue des patois gallo-romans
28  11936293           Château de la Motte-au-Bois
30  13586687  Die Entstehung des socialen Problems

[20 rows x 2 columns]

✓ Exported verification list to: 